# Ticket GOLD-001: Build match_performance_scores
*Goal*: `CREATE OR REPLACE TABLE marvel_rivals.gold.match_performance_scores` — one row per player-hero per match, with role-weighted performance score normalized to 0-100.

Conceptually:
```markdown
performance_score = (
    w1 * kills_per10 +
    w2 * deaths_per10 (inverted — more deaths = lower score) +
    w3 * assists_per10 +
    w4 * damage_per10 +
    w5 * healing_per10 +
    w6 * damage_taken_per10
)
```

Where the weights w1-w5 changed based on role:
```markdown
Duelist weights:
    kills_per10       → high weight (0.30)
    damage_per10      → high weight (0.30)
    deaths_per10      → medium penalty (-0.20)
    assists_per10     → low weight (0.10)
    healing_per10     → zero weight (0.00)
    damage_taken_per10→ zero weight (0.10)

Vanguard weights:
    damage_taken_per10→ high weight (0.35)  ← absorbing damage is the job
    kills_per10       → low weight (0.10)
    damage_per10      → medium weight (0.20)
    deaths_per10      → medium penalty (-0.20)
    assists_per10     → medium weight (0.15)
    healing_per10     → zero weight (0.00)

Strategist weights:
    healing_per10     → high weight (0.40)  ← healing is the job
    assists_per10     → medium weight (0.25)
    deaths_per10      → medium penalty (-0.20)
    kills_per10       → low weight (0.05)
    damage_per10      → low weight (0.10)
    damage_taken_per10→ zero weight (0.00)
```

The normalizations step converts the raw weighted sum into 0-100 using min-max scaling
`normalized_score = (raw_score - min_score) / (max_score - min_score) * 100`

This means the best performer in the dataset always gets ~100 and the worst gets ~0, making scores comparable across matches and heroes regardless of the raw numbers. 

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.gold.match_performance_scores AS 
WITH base AS (
    SELECT * FROM marvel_rivals.silver.player_hero_stats
),
weighted AS (
    SELECT *, 
        CASE hero_role
            WHEN 'Duelist' THEN
                (kills_per_10_minutes * 0.30) +
                (damage_per_10_minutes * 0.30) +
                (assists_per_10_minutes * 0.10) +
                (damage_taken_per_10_minutes * 0.10) +
                (deaths_per_10_minutes * -0.20)
            WHEN 'Vanguard' THEN
                (damage_taken_per_10_minutes * 0.35) +
                (damage_per_10_minutes * 0.20) +
                (assists_per_10_minutes * 0.15) +
                (kills_per_10_minutes * 0.10) +
                (deaths_per_10_minutes * -0.20)
            WHEN 'Strategist' THEN
                (healing_per_10_minutes * 0.40) +
                (assists_per_10_minutes * 0.25) +
                (kills_per_10_minutes * 0.05) +
                (damage_per_10_minutes * 0.10) +
                (deaths_per_10_minutes * -0.20)
            ELSE
                (kills_per_10_minutes * 0.30) +
                (damage_per_10_minutes * 0.30) +
                (assists_per_10_minutes * 0.10) +
                (damage_taken_per_10_minutes * 0.10) +
                (deaths_per_10_minutes * -0.20)
        END AS raw_performance_score,
        CASE 
            WHEN hero_role LIKE '%/%' THEN 'Duelist'  -- Deadpool default
            ELSE hero_role 
        END AS role_weight_applied
    FROM base
),
normalized AS (
    SELECT *,
    MIN(raw_performance_score) OVER () AS min_score,
    MAX(raw_performance_score) OVER () AS max_score
    FROM weighted
),
scored AS (
    SELECT *,
    ROUND((raw_performance_score - min_score) / NULLIF(max_score - min_score, 0) * 100, 2) AS performance_score_0_100
    FROM normalized
)
SELECT * EXCEPT(ingestion_timestamp), CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM scored


In [0]:
-- Verify row count
SELECT COUNT(*) as total_rows FROM marvel_rivals.gold.match_performance_scores;

-- Check your own performance scores
SELECT 
    nick_name,
    hero_name,
    hero_role,
    kills_per_10_minutes,
    healing_per_10_minutes,
    damage_per_10_minutes,
    add_score,
    performance_score_0_100,
    role_weight_applied
FROM marvel_rivals.gold.match_performance_scores
WHERE nick_name = 'anonimis00'
ORDER BY performance_score_0_100 DESC
LIMIT 10;

In [0]:
-- Verify row count
SELECT COUNT(*) as total_rows FROM marvel_rivals.gold.match_performance_scores;

-- Check Boorex's own performance scores
SELECT 
    nick_name,
    hero_name,
    hero_role,
    kills_per_10_minutes,
    healing_per_10_minutes,
    damage_per_10_minutes,
    add_score,
    performance_score_0_100,
    role_weight_applied
FROM marvel_rivals.gold.match_performance_scores
WHERE nick_name = 'Boorex'
ORDER BY performance_score_0_100 DESC
LIMIT 10;

# Ticket GOLD-002: Build squad_performance_summary
*Goal* `CREATE OR REPLACE TABLE marvel_rivals.gold.squad_performance_summary` - one row per squad member, aggregating their performance across all matches.

Requirements - compute these aggregations:
```sql
COUNT(DISTINCT match_uid)           AS total_matches,
ROUND(AVG(performance_score_0_100), 2) AS avg_performance_score,
ROUND(AVG(add_score), 2)            AS avg_add_score,
ROUND(SUM(CASE WHEN is_win = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS win_rate_pct,
ROUND(AVG(kills_per_10_minutes), 2) AS avg_kills_per10,
ROUND(AVG(deaths_per_10_minutes), 2) AS avg_deaths_per10,
ROUND(AVG(healing_per_10_minutes), 2) AS avg_healing_per10,
ROUND(AVG(damage_per_10_minutes), 2) AS avg_damage_per10,
MODE(hero_name)                     AS most_played_hero,
MODE(hero_role)                     AS most_played_role
```
Group by: `nick_name`
Source: `marvel_rivals.gold.match_performance_scores`
Note on `MODE()`: Spark SQL doesn't have a native `MODE()` function. Instead we'll use `FIRST()` 
OR
Or the proper approach with a window function — find the most frequent `hero_name` per `nick_name` using COUNT + ROW_NUMBER().


In [0]:
CREATE OR REPLACE TABLE marvel_rivals.gold.squad_performance_summary AS
WITH hero_counts AS (
    SELECT
        nick_name,
        hero_name,
        hero_role,
        COUNT(*) AS hero_appearances,
        ROW_NUMBER() OVER (
            PARTITION BY nick_name 
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM marvel_rivals.gold.match_performance_scores
    GROUP BY nick_name, hero_name, hero_role
),
most_played AS (
    SELECT nick_name, hero_name AS most_played_hero, hero_role AS most_played_role
    FROM hero_counts WHERE rn = 1
),
aggregated AS (
    SELECT
        nick_name,
        COUNT(DISTINCT match_uid) AS total_matches,
        ROUND(AVG(performance_score_0_100), 2) AS avg_performance_score,
        ROUND(AVG(add_score), 2) AS avg_add_score,
        ROUND(AVG(performance_score_0_100) - AVG(add_score), 2) AS fairness_gap,
        ROUND(AVG(is_win) * 100, 2) AS win_rate_pct,
        ROUND(AVG(kills_per_10_minutes), 2) AS avg_kills_per10,
        ROUND(AVG(deaths_per_10_minutes), 2) AS avg_deaths_per10,
        ROUND(AVG(healing_per_10_minutes), 2) AS avg_healing_per10,
        ROUND(AVG(damage_per_10_minutes), 2)  AS avg_damage_per10
    FROM marvel_rivals.gold.match_performance_scores
    GROUP BY nick_name
)
SELECT 
    a.*,
    mp.most_played_hero,
    mp.most_played_role,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM aggregated a
JOIN most_played mp ON a.nick_name = mp.nick_name

In [0]:
-- Check how many rows per nick_name in the summary
SELECT nick_name, COUNT(*) as row_count
FROM marvel_rivals.gold.squad_performance_summary
GROUP BY nick_name
ORDER BY row_count DESC;

In [0]:
-- Check if most_played CTE has duplicates
WITH hero_counts AS (
    SELECT
        nick_name,
        hero_name,
        hero_role,
        COUNT(*) AS hero_appearances,
        ROW_NUMBER() OVER (
            PARTITION BY nick_name 
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM marvel_rivals.gold.match_performance_scores
    GROUP BY nick_name, hero_name, hero_role
)
SELECT nick_name, COUNT(*) as rn1_count
FROM hero_counts
WHERE rn = 1
GROUP BY nick_name
HAVING COUNT(*) > 1;

In [0]:
-- Check row count in aggregated CTE
SELECT nick_name, COUNT(*) as row_count
FROM marvel_rivals.gold.match_performance_scores
GROUP BY nick_name
ORDER BY row_count DESC;

In [0]:
SELECT COUNT(DISTINCT nick_name) 
FROM marvel_rivals.gold.match_performance_scores;

In [0]:
SELECT * FROM marvel_rivals.gold.squad_performance_summary
ORDER BY avg_performance_score DESC

In [0]:
SELECT DISTINCT nick_name, player_uid
FROM marvel_rivals.gold.match_performance_scores
WHERE nick_name IN ('KingCyrex', 'StarGamingZ250')

In [0]:
CREATE OR REPLACE TABLE marvel_rivals.gold.squad_performance_summary AS
WITH hero_counts AS (
    SELECT
        CASE player_uid
            WHEN 676793468  THEN 'Player_You'
            WHEN 1684171792 THEN 'Player_A'
            WHEN 2097825933 THEN 'Player_A'
            WHEN 88596421   THEN 'Player_B'
            WHEN 1976310301 THEN 'Player_B'
            WHEN 844873320  THEN 'Player_C'
            WHEN 1811103264 THEN 'Player_D'
            WHEN 205883953  THEN 'Player_E'
        END AS canonical_name,
        hero_name,
        hero_role,
        COUNT(*) AS hero_appearances,
        ROW_NUMBER() OVER (
            PARTITION BY CASE player_uid
                WHEN 676793468  THEN 'Player_You'
                WHEN 1684171792 THEN 'Player_A'
                WHEN 2097825933 THEN 'Player_A'
                WHEN 88596421   THEN 'Player_B'
                WHEN 1976310301 THEN 'Player_B'
                WHEN 844873320  THEN 'Player_C'
                WHEN 1811103264 THEN 'Player_D'
                WHEN 205883953  THEN 'Player_E'
            END
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM marvel_rivals.gold.match_performance_scores
    WHERE player_uid IN (
        676793468, 1684171792, 2097825933,
        88596421, 1976310301, 844873320,
        1811103264, 205883953
    )
    GROUP BY canonical_name, hero_name, hero_role
),
most_played AS (
    SELECT canonical_name, hero_name AS most_played_hero, hero_role AS most_played_role
    FROM hero_counts WHERE rn = 1
),
aggregated AS (
    SELECT
        CASE player_uid
            WHEN 676793468  THEN 'Player_You'
            WHEN 1684171792 THEN 'Player_A'
            WHEN 2097825933 THEN 'Player_A'
            WHEN 88596421   THEN 'Player_B'
            WHEN 1976310301 THEN 'Player_B'
            WHEN 844873320  THEN 'Player_C'
            WHEN 1811103264 THEN 'Player_D'
            WHEN 205883953  THEN 'Player_E'
        END AS canonical_name,
        COUNT(DISTINCT match_uid) AS total_matches,
        ROUND(AVG(performance_score_0_100), 2) AS avg_performance_score,
        ROUND(AVG(add_score), 2) AS avg_add_score,
        ROUND(AVG(performance_score_0_100) - AVG(add_score), 2) AS fairness_gap,
        ROUND(AVG(is_win) * 100, 2) AS win_rate_pct,
        ROUND(AVG(kills_per_10_minutes), 2) AS avg_kills_per10,
        ROUND(AVG(deaths_per_10_minutes), 2) AS avg_deaths_per10,
        ROUND(AVG(healing_per_10_minutes), 2) AS avg_healing_per10,
        ROUND(AVG(damage_per_10_minutes), 2)  AS avg_damage_per10
    FROM marvel_rivals.gold.match_performance_scores
    WHERE player_uid IN (
        676793468, 1684171792, 2097825933,
        88596421, 1976310301, 844873320,
        1811103264, 205883953
    )
    GROUP BY canonical_name
)
SELECT 
    a.*,
    mp.most_played_hero,
    mp.most_played_role,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM aggregated a
JOIN most_played mp ON a.canonical_name = mp.canonical_name

In [0]:
SELECT * FROM marvel_rivals.gold.squad_performance_summary
ORDER BY avg_performance_score DESC